# 01 — MonReader Dataset, Source Reconstruction and Evaluation Regimes

**Research question:** How can image filenames and reconstructed video sources support transparent, leakage-aware evaluation of page-turn classification?

This notebook builds the dataset audit, checks chronology, and generates explicit O, S, T and ST partitions. It also preserves a distinction between the original reported results and any newly executed partitions.

**Inputs:** Original MonReader dataset extracted under `data/raw/`, or configured with `MONREADER_DATA_ROOT`. **Outputs:** Canonical manifest, split CSVs, class/source summaries, and an illustrative regime figure. Run this from the repository root.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPyImage
import src as mr

# Accept both ZIP extraction layouts:
# data/raw/training/... and data/raw/images/training/...
ROOT = Path.cwd().resolve()
if not (ROOT/'results'/'dataset_regime_summary.csv').exists():
    raise RuntimeError('Launch Jupyter from the monreader-proofread repository root.')
settings = mr.get_settings(ROOT)
raw = Path(settings['raw'])
if (raw/'training').is_dir() and (raw/'testing').is_dir():
    dataset_root = raw
elif (raw/'images'/'training').is_dir() and (raw/'images'/'testing').is_dir():
    dataset_root = raw/'images'
else:
    raise FileNotFoundError(
        f'Original dataset missing from {raw}. Complete extraction first. '
        'Expected training/ and testing/ either here or under images/.'
    )
settings['raw'] = str(dataset_root.resolve())
device = mr.setup(settings['seed'])
(ROOT/'generated').mkdir(parents=True,exist_ok=True)
mr.write_json(ROOT/'generated'/'experiment_settings.json',settings)
print('Repository:',ROOT)
print('Detected dataset:',settings['raw'])
print('Output:',settings['out'])
print('Device:',device)


## Data and scientific provenance

The original dataset comprises RGB images labelled `flip` and `notflip`. File stems encode a zero-padded reconstructed source identifier and an observed frame number. The class directory is part of the canonical source key because numeric source identifiers are reused.

**Provenance:** The accepted historical O/S/T/ST results belong to their original frozen split populations, not to any newly executed partition. Their held-out sizes were O=597, S=610, T=301 and ST=121. This notebook reconstructs the original source-safe and chronological assignments while keeping the recorded historic scores immutable. The 489 earlier frames of the original 24 unseen ST sources remain context only. The researcher should review the resulting counts before claiming any historical score has been replicated.

The source identifiers and frame order do not establish capture dates, independent recording sessions, or elapsed-time gaps.


## Inspect and decode the original filenames

A name such as `0065_000000015.jpg` contains two candidate fields: recording/environment **0065** and frame index **15**. The labels are obtained from the actual enclosing `flip`/`notflip` directory, not guessed from pixels. Numeric identifiers can recur across classes; cross-label collisions are reported, and `label::video_id` is treated as a reconstructed grouping hypothesis, subject to verification against recording provenance. Invalid filename patterns cause a hard failure rather than silently dropping images.

In [ ]:
manifest = mr.discover_manifest(settings['raw'])
expected_classes={'flip':1452,'notflip':1537}
expected_supplied={'training':2392,'testing':597}
if len(manifest)!=2989 or manifest.label.value_counts().to_dict()!=expected_classes:
    raise AssertionError('Dataset does not match original 2,989-frame class counts.')
if manifest.supplied_split.value_counts().to_dict()!=expected_supplied:
    raise AssertionError('Original supplied train/test folder counts do not match.')
if manifest.source_key.nunique()!=117:
    raise AssertionError('Original dataset must contain 117 reconstructed source videos.')
print('Verified original frame count, class balance, source count, and folder split.')
display(manifest[['relative_path','label','video_id','frame_number','source_key']].head(12))
audit = mr.audit_manifest(manifest, settings['out'])
display(pd.DataFrame([audit]).T.rename(columns={0:'observed'}))
manifest.to_csv(Path(settings['out'])/'canonical_manifest.csv',index=False)


## Evaluate source structure and chronological quality

We now inspect **measured** class counts and frames per reconstructed source. A large number of frames per source implies correlated observations; it does not by itself establish duplicated pixels. The collision and duplicate-pixel reports were calculated directly above. The temporal split will reject sources with insufficient frames or repeated chronology where strict ordering cannot be verified.

In [ ]:
import matplotlib.pyplot as plt
frames_per_source=manifest.groupby('source_key').size().rename('frames')
display(pd.DataFrame({'class_frames':manifest.label.value_counts()}))
display(frames_per_source.describe().to_frame())
fig,ax=plt.subplots(figsize=(8,4),layout='constrained')
ax.hist(frames_per_source.to_numpy(),bins=min(20,len(frames_per_source)))
ax.set(title='Observed frames per reconstructed source',xlabel='Frames',ylabel='Number of sources')
figure_dir=Path(settings['out'])/'figures'
figure_dir.mkdir(parents=True,exist_ok=True)
fig.savefig(figure_dir/'frames_per_source.png',dpi=160,bbox_inches='tight')
plt.show()


## Construct O, S, T and ST without looking at model performance

**O** preserves the supplied 597-frame test set, taking validation only from the originally supplied 2,392 training frames. **S** makes source-disjoint partitions. **T** assigns early/middle/late frames chronologically within every source. **ST** separates unseen sources first; their early frames are explicitly labelled `context_unseen_early` and are never used for fitting or selection. Exact counts are derived, **not entered as targets**. Tests verify sample accounting, class coverage and the restrictions appropriate to each regime. Every split is saved and SHA-256 fingerprinted before training.

In [ ]:
splits=mr.make_splits(manifest,seed=settings['seed'])
expected_counts={
    'O':{'train':2152,'validation':240,'test':597},
    'S':{'train':2147,'validation':232,'test':610},
    'T':{'train':2387,'validation':301,'test':301},
    'ST':{'train':1904,'validation':475,'context_unseen_early':489,'test':121},
}
for regime, df in splits.items():
    observed=df.role.value_counts().to_dict()
    if observed != expected_counts[regime]:
        raise AssertionError(
            f'{regime}: got {observed} instead of {expected_counts[regime]}. '
            'Do not proceed with a mismatched historical cohort.'
        )
    print(f'\n{regime}: confirmed {observed}')
    display(pd.crosstab(df.role,df.label))
assert splits['ST'].loc[splits['ST'].role=='test','source_key'].nunique()==24
split_summary=mr.save_splits(splits,settings['out'])
print('Historical split counts, including the 24-source ST holdout, verified.')
display(split_summary)


## Regime figure and research handoff

Real image panels are selected by their actual assigned roles. The figure is **illustrative** of the four designs, not a second source of truth for split membership. Notebooks 02–05 will consume the saved manifest and split CSVs, so the entire experiment can be reopened in a new Jupyter kernel.

In [ ]:
figure = mr.plot_regime_examples(
    splits, Path(settings['out'])/'figures'/'evaluation_regime_examples.png'
)
display(IPyImage(filename=str(figure)))
print('Regime illustration generated from the current split membership:',figure)


### Notebook 01 interpretation and handoff

Inspect the observed class counts, source lengths, split membership and source exclusion checks. O mixes correlated frames; S tests new sources; T tests later frames within familiar sources; and ST combines source exclusion with later-position evaluation. These partitions answer different questions and are not interchangeable. Notebook 02 will tune Scratch CNNs using validation data only.


## Historical split table — preserved reference

This table is loaded from the original submitted results. It is an **archived observation**, distinct from the newly generated split summary above.

In [ ]:
historical_splits=mr.read_archived_result_table(ROOT/'results'/'dataset_regime_summary.csv')
display(historical_splits[['regime','train_images','validation_images','test_images','test_source_videos']])